In [1]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent)

print(Path.cwd())

d:\private\ai-research-paper-assistant


In [2]:
import os
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from src.ingestion.parser import SimpleLoader
from src.ingestion.chunker import RecursiveChunk
from src.embedder.ollama_embedder import OllamaEmbedder
from src.database.pgvector_storage import PGVectorStore
from src.utils.helpers import IngestionPipeline
from src.generation.ollama_llm import LLM
from src.reranker.cross_encoder_reranker import CrossEncoderReRanker
from src.utils.helpers import format_context, RAGChain, IngestionPipeline

c:\Users\luann\miniconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_one_json(path: str):
    path = Path(path)
    if not path.exists():
        raise ValueError("Invalid path")
    with open(path, "r", encoding="utf-8") as f:
        contents = json.load(f)
    return contents

def load_dir_json(dir_path: str):
    path = Path(dir_path)
    if not path.exists():
        raise ValueError("Invalid path")
    all_files = path.glob("*.json")
    all_contents = []
    for file in tqdm(all_files, desc="Processing"):
        all_contents.append(load_one_json(file))
    return all_contents

In [4]:
all_contents = load_dir_json(r"D:\private\ai-research-paper-assistant\data\eval")
print(len(all_contents))
print(all_contents[:10])

Processing: 5it [00:00, 655.59it/s]

5
[{'dataset_name': 'arcface_additive_angular_margin_loss_rag_eval', 'samples': [{'id': 'q001', 'difficulty': 'Easy', 'question_type': 'Definition', 'question': 'What is the geometric interpretation of the additive angular margin penalty in ArcFace?', 'reference_answer': 'The additive angular margin penalty in ArcFace corresponds directly to the geodesic distance margin penalty on the normalized hypersphere.', 'evidence': [{'page': 3, 'section': '3.1 ArcFace', 'evidence_text': 'Since the proposed additive angular margin penalty is equal to the geodesic distance margin penalty in the normalized hypersphere, we name our method as ArcFace.'}]}, {'id': 'q002', 'difficulty': 'Easy', 'question_type': 'Architecture', 'question': 'What specific layer structure is applied after the final convolutional layer to generate the 512-D embedding feature in ArcFace?', 'reference_answer': 'After the last convolutional layer, the architecture utilizes a BN-Dropout-FC-BN structure to produce the final 512

In [5]:
loader = SimpleLoader()
chunker = RecursiveChunk(chunk_size=1200, chunk_overlap=100)
embedder = OllamaEmbedder(dimensions=2046, num_ctx=8192, num_gpu=1)
repo = PGVectorStore()
llm = LLM(temperature=0.1, model_name="qwen3.5:2b", num_ctx=8192, num_gpu=1, reasoning=False)
reranker = CrossEncoderReRanker()
rag = RAGChain(llm, repo, embedder, reranker)

ingestor = IngestionPipeline(loader=loader, chunker=chunker, embedder=embedder, repository=repo)

In [ ]:
papers_path = r"D:\private\ai-research-paper-assistant\papers"
ingestor.ingest_dir(papers_path)

In [19]:
all_chunks = repo.get_all_chunks()
cnt = 0
for chunk in all_chunks:
    print(chunk["metadata"].get("source", ""))
        

D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.0264

In [7]:
# eval_questions = [
#         "Can you provide a concise description of the TinyLlama model?",
#         "I would like to know the speed optimizations that TinyLlama has made.",
#         "Why TinyLlama uses Grouped-query Attention?",
#         "Is the TinyLlama model open source?",
#         "Tell me about starcoderdata dataset",
#     ]

# eval_answers = [
#         "TinyLlama is a compact 1.1B language model pretrained on around 1 trillion tokens for approximately 3 epochs. Building on the architecture and tokenizer of Llama 2, TinyLlama leverages various advances contributed by the open-source community (e.g., FlashAttention), achieving better computational efficiency. Despite its relatively small size, TinyLlama demonstrates remarkable performance in a series of downstream tasks. It significantly outperforms existing open-source language models with comparable sizes.",
#         "During training, our codebase has integrated FSDP to leverage multi-GPU and multi-node setups efficiently. Another critical improvement is the integration of Flash Attention, an optimized attention mechanism. We have replaced the fused SwiGLU module from the xFormers (Lefaudeux et al., 2022) repository with the original SwiGLU module, further enhancing the efficiency of our codebase. With these features, we can reduce the memory footprint, enabling the 1.1B model to fit within 40GB of GPU RAM.",
#         "To reduce memory bandwidth overhead and speed up inference, we use grouped-query attention in our model. We have 32 heads for query attention and use 4 groups of key-value heads. With this technique, the model can share key and value representations across multiple heads without sacrificing much performance",
#         "Yes, TinyLlama is open-source",
#         "This dataset was collected to train StarCoder (Li et al., 2023), a powerful opensource large code language model. It comprises approximately 250 billion tokens across 86 programming languages. In addition to code, it also includes GitHub issues and text-code pairs that involve natural languages.",
#     ]

eval_questions = []
eval_answers = []

for doc in all_contents:
    sample = doc["samples"]
    for q in sample:
        eval_questions.append(q["question"])
        eval_answers.append(q["reference_answer"])

print(len(eval_questions))
print(len(eval_answers))

100
100


In [8]:
all_chunks = repo.get_all_chunks()
all_chunks[0]

{'id': 5777,
 'document_id': '1506.02640v5',
 'content': 'You Only Look Once:\nUniﬁed, Real-Time Object Detection\nJoseph Redmon∗, Santosh Divvala∗†, Ross Girshick¶, Ali Farhadi∗†\nUniversity of Washington∗, Allen Institute for AI†, Facebook AI Research¶\nhttp://pjreddie.com/yolo/\nAbstract\nWe present YOLO, a new approach to object detection.\nPrior work on object detection repurposes classiﬁers to per-\nform detection. Instead, we frame object detection as a re-\ngression problem to spatially separated bounding boxes and\nassociated class probabilities. A single neural network pre-\ndicts bounding boxes and class probabilities directly from\nfull images in one evaluation. Since the whole detection\npipeline is a single network, it can be optimized end-to-end\ndirectly on detection performance.\nOur uniﬁed architecture is extremely fast. Our base\nYOLO model processes images in real-time at 45 frames\nper second. A smaller version of the network, Fast YOLO,\nprocesses an astounding 15

In [9]:
t = rag.reranker.rerank("WHAT is arcface", rag.hybrid_searcher.retrieve("WHAT is arcface"))
t

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2580.24it/s]


[{'id': 5862,
  'document_id': '1801.07698v4',
  'content': 'JOURNAL OF LATEX CLASS FILES, VOL. 14, NO. 8, AUGUST 2015 1\nArcFace: Additive Angular Margin Loss for Deep\nFace Recognition\nJiankang Deng, Jia Guo, Jing Y ang, Niannan Xue, Irene Kotsia, and Stefanos Zafeiriou\nAbstract—Recently, a popular line of research in face recognition is adopting margins in the well-established softmax loss function to\nmaximize class separability. In this paper, we ﬁrst introduce an Additive Angular Margin Loss (ArcFace), which not only has a clear\ngeometric interpretation but also signiﬁcantly enhances the discriminative power. Since ArcFace is susceptible to the massive label\nnoise, we further propose sub-center ArcFace, in which each class contains K sub-centers and training samples only need to be\nclose to any of theK positive sub-centers. Sub-center ArcFace encourages one dominant sub-class that contains the majority of clean\nfaces and non-dominant sub-classes that include hard or noisy f

In [10]:
evaluation_data = []

for question, reference in tqdm(zip(eval_questions, eval_answers), desc="Processing", total=len(eval_questions)):
    context_rrf = rag.hybrid_searcher.retrieve(
        question,
        top_k=10
        )

    context_reranked = rag.reranker.rerank(
        question,
        context_rrf,
        top_k=5
        )

        # RAGAS cần list[str]
    retrieved_contexts = [doc["content"] for doc in context_reranked]

        # LLM của RAG cần formatted string
    context = format_context(
        context_reranked
        )

    message = rag.prompt.invoke({
        "context": context,
        "question": question
        })

    response = rag.llm.generate(
         message
         ).content

    evaluation_data.append({
        "user_input": question,
        "reference": reference,
        "retrieved_contexts": retrieved_contexts,
        "response": response,
        })

# Query/user_input, predict_chunk_ids(document_id), reference_chunk_ids(document_id), reference, retrieved_contexts, response

Processing: 100%|██████████| 100/100 [39:20<00:00, 23.61s/it]


In [12]:
print(evaluation_data[0]["user_input"])
print()

print("REFERENCE:")
print(evaluation_data[0]["reference"])
print()

print("CONTEXTS:")
print(evaluation_data[0]["retrieved_contexts"])
print(type(evaluation_data[0]["retrieved_contexts"]))
print(len(evaluation_data[0]["retrieved_contexts"]))

print("\nRESPONSE:")
print(evaluation_data[0]["response"])

What is the geometric interpretation of the additive angular margin penalty in ArcFace?

REFERENCE:
The additive angular margin penalty in ArcFace corresponds directly to the geodesic distance margin penalty on the normalized hypersphere.

CONTEXTS:
['JOURNAL OF LATEX CLASS FILES, VOL. 14, NO. 8, AUGUST 2015 1\nArcFace: Additive Angular Margin Loss for Deep\nFace Recognition\nJiankang Deng, Jia Guo, Jing Y ang, Niannan Xue, Irene Kotsia, and Stefanos Zafeiriou\nAbstract—Recently, a popular line of research in face recognition is adopting margins in the well-established softmax loss function to\nmaximize class separability. In this paper, we ﬁrst introduce an Additive Angular Margin Loss (ArcFace), which not only has a clear\ngeometric interpretation but also signiﬁcantly enhances the discriminative power. Since ArcFace is susceptible to the massive label\nnoise, we further propose sub-center ArcFace, in which each class contains K sub-centers and training samples only need to be\nclose

In [14]:
import json
with open(r"data\eval_data.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_data, f)

In [ ]:
df = pd.DataFrame(evaluation_data)
df.to_csv("eval_data.csv")